# Annotate Variants with LOFTEE

- Date Modified: 6/20/2026

- Description: Uses the LOFTEE module from VEP to annotate variants of interest as high or low confidence LoF


In [ ]:
# set work directory
WORK_DIR = "vep_analysis"
import pandas as pd

In [ ]:
%%bash
# Annotate UKB LOF (replace input and output names for other data)
# run vep LOFTEE
export PERL5LIB=$PERL5LIB:${VEP_CACHEDIR}/Plugins/loftee_GRCh38
module load VEP/111
vep \
-i input/UKB_WGS_ITSN1_vep_input.vcf.gz \ 
-o output/UKB_WGS_vep_out \
--offline \
--cache \
--fork 6 \
--force_overwrite \
--flag_pick_allele \
--dir ${VEP_CACHEDIR} \
--species human \
--assembly GRCh38 \
--fasta ${VEP_CACHEDIR}/GRCh38.fa \
--everything \
--plugin "CADD,${VEP_CACHEDIR}/CADD_1.4_GRCh38_whole_genome_SNVs.tsv.gz,${VEP_CACHEDIR}/CADD_1.4_GRCh38_InDels.tsv.gz" \
--dir_plugins ${VEP_CACHEDIR}/Plugins/loftee_GRCh38 \
--plugin LoF,loftee_path:${VEP_CACHEDIR}/Plugins/loftee_GRCh38,\
human_ancestor_fa:${VEP_CACHEDIR}/Plugins/loftee_GRCh38/human_ancestor.fa.gz,\
conservation_file:${VEP_CACHEDIR}/Plugins/loftee_GRCh38/loftee.sql,\
gerp_bigwig:${VEP_CACHEDIR}/Plugins/loftee_GRCh38/gerp_conservation_scores.homo_sapiens.GRCh38.bw \
--plugin SpliceAI,snv=${VEP_CACHEDIR}/spliceai_scores.masked.snv.hg38.vcf.gz,indel=${VEP_CACHEDIR}/spliceai_scores.masked.indel.hg38.vcf.gz

[+] Loading VEP 111 on cn0038 
[+] Loading singularity  4.3.7  on cn0038 


In [11]:
# function to add VEP annotations to glm output
def merge_vep_glm(vep_out_path, databank=""):
    # read in vep outfile
    vep_df = pd.read_csv(f"{vep_out_path}", sep="\t", skiprows=111)
    # filter to those chosen by pick and HC Loftee
    filtered_lof_vep = vep_df[vep_df["Extra"].str.contains("PICK=1") & vep_df["Extra"].str.contains("LoF=HC") & vep_df["Gene"].str.contains("ENSG00000205726")]
    filtered_lof_vep
    # merged filtered vep data with results file to see what ancestries they fall in

    results = pd.read_csv(f"{WORK_DIR}/{databank}.csv")

    # save merged dataset to file
    filtered_lof_vep.merge(results, left_on="#Uploaded_variation", right_on="SNP")[["SNP", "Consequence_x", 
                                                                                    "Consequence_y", "Ancestry", "F_A", "F_U",
                                                                                    "Existing_variation", "Extra"]].to_csv(f"{databank}_LOFTEE_annotation.csv", index=False)
    

In [ ]:
# get loftee annotations for ukb, aou and gp2
data = ["AOU_WGS", "UKB_WGS", "GP2_WGS"]
for bio in data:
    merge_vep_glm(f"output/{bio}_vep_out", databank=bio)